In [ ]:
import cProfile
import pstats
import io
import time
import traceback
import os, time
import csv
import numpy as np
from pathlib import Path
from datetime import timedelta
from fastai.callback.all import *
from fastai.callback.tracker import SaveModelCallback as _SaveModelCallback

In [ ]:
class EpochTracker(Callback):
    """
    Tracks epochs completed, saves training log with all metrics.
    Persists state to disk for crash recovery.
    """
    order = 60
    _state_path  = Path('training_stats/epoch_tracker.json')
    _log_path    = Path('training_stats/training_log.csv')
    _log_fields  = [
        'epoch', 'train_loss', 'valid_loss',
        'sisnr_db', 'noise_removed_%',
        'sdr_improvement_db', 'seg_snr_improvement_db', 'log_spectral_dist',
        'lsd_optimal_gain', 'current_lr', 'lr_max', 'epoch_elapsed_time'
    ]

    def __init__(self, total_epochs=300):
        self.total_epochs = total_epochs
        self._start_time  = None
        # restore persisted state
        try:
            if self._state_path.exists():
                s = json.loads(self._state_path.read_text())
                self.epochs_done   = s.get('epochs_done', 0)
                self.saved_lr_max  = s.get('saved_lr_max', 1.4e-4)
            else:
                self.epochs_done  = 0
                self.saved_lr_max = 1.4e-4
        except Exception:
            self.epochs_done  = 0
            self.saved_lr_max = 1.4e-4

    def before_fit(self):
        self._state_path.parent.mkdir(exist_ok=True)
        self._log_path.parent.mkdir(exist_ok=True)
        # write CSV header if new file
        if not self._log_path.exists():
            with open(self._log_path, 'w', newline='') as f:
                csv.DictWriter(f, fieldnames=self._log_fields).writeheader()

    def before_epoch(self):
        self._start_time = time.time()
        set_epoch_seed(self.epochs_done)

    def after_epoch(self):
        # Skip ghost epochs - real epochs take more than a few seconds
        # A completed epoch at bs=2 with 48000 samples takes ~1.5 hours
        # Any epoch under 5 seconds is a failed/cancelled epoch
        elapsed = time.time() - self._start_time

        if elapsed < 5.0:
            return

        elapsed_str = str(timedelta(seconds=int(elapsed)))

        # collect metric values by name for robustness
        metric_vals = {}
        if hasattr(self.learn, 'recorder') and self.learn.recorder.values:
            names  = ['train_loss', 'valid_loss'] + \
                     [m.name for m in self.learn.metrics]
            values = self.learn.recorder.values[-1]
            for n, v in zip(names, values):
                metric_vals[n] = v

        current_lr = self.learn.opt.hypers[-1].get('lr', float('nan'))
        lr_max = self.saved_lr_max

        row = {
            'epoch': self.epochs_done + 1,
            'train_loss': metric_vals.get('train_loss', ''),
            'valid_loss': metric_vals.get('valid_loss', ''),
            'sisnr_db': metric_vals.get('sisnr_db', ''),
            'noise_removed_%': metric_vals.get('noise_removed_%', ''),
            'sdr_improvement_db': metric_vals.get('sdr_improvement_db', ''),
            'seg_snr_improvement_db': metric_vals.get('seg_snr_improvement_db', ''),
            'log_spectral_dist': metric_vals.get('log_spectral_dist', ''),
            'lsd_optimal_gain': metric_vals.get('lsd_optimal_gain', ''),
            'current_lr': current_lr,
            'lr_max': lr_max,
            'epoch_elapsed_time': elapsed_str,
        }

        with open(self._log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self._log_fields).writerow(row)

        self.epochs_done += 1
        self._save_state()

    def _save_state(self):
        try:
            self._state_path.write_text(json.dumps({
                'epochs_done':  self.epochs_done,
                'saved_lr_max': self.saved_lr_max,
                'total_epochs': self.total_epochs,
            }))
        except Exception as e:
            print(f"[EpochTracker] state save failed: {e}")

    @property
    def total_epochs(self): return self._total_epochs
    @total_epochs.setter
    def total_epochs(self, v): self._total_epochs = v


In [ ]:
class SISNRDiagnostic(Callback):
    """
    Per-sample SI-SNR diagnostic for fixed validation samples.
    Also captures pred power, target power, power ratio, and all
    metrics (SDRi, SegSNR, LSD) for the same fixed samples.
    Writes to training_stats/sisnr_diagnostic.csv.
    """
    order = 70
    _log_path = Path('training_stats/sisnr_diagnostic.csv')
    _log_fields = [
        'epoch',
        'sisnr_sample_0', 'sisnr_sample_1',
        'sisnr_sample_2', 'sisnr_sample_3',
        'sisnr_mean',
        'sdr_sample_0', 'sdr_sample_1',
        'sdr_mean',
        'seg_snr_sample_0', 'seg_snr_sample_1',
        'seg_snr_mean',
        'lsd_sample_0', 'lsd_sample_1',
        'lsd_mean',
        'pred_min', 'pred_max',
        'pred_power_mean', 'targ_power_mean', 'power_ratio',
    ]

    def __init__(self, n_samples=4):
        self.n_samples = n_samples
        self._fixed_batch = None  # captured once from first validation batch

    def before_fit(self):
        self._log_path.parent.mkdir(exist_ok=True)
        if not self._log_path.exists():
            with open(self._log_path, 'w', newline='') as f:
                csv.DictWriter(f, fieldnames=self._log_fields).writeheader()
        self._fixed_batch = None  # reset on each fit call

    def after_validate(self):
        if self._fixed_batch is None:
            return
        noisy_b, clean_b, pred_b = self._fixed_batch
        n = min(self.n_samples, pred_b.shape[0])
        eps = 1e-8

        def si_snr_single(pred, targ):
            targ_zm = targ - targ.mean()
            pred_zm = pred - pred.mean()
            alpha = (targ_zm * pred_zm).sum() / (targ_zm.pow(2).sum() + eps)
            s = (alpha * targ_zm).pow(2).sum()
            noise = (pred_zm - alpha * targ_zm).pow(2).sum()
            return (10 * torch.log10((s + eps) / (noise + eps))).item()

        def sdr_single(pred, ref):
            return (10 * torch.log10(
                ref.pow(2).sum() / ((pred - ref).pow(2).sum() + eps) + eps
            )).item()

        def seg_snr_single(pred, ref):
            frame = 320
            T = pred.shape[-1] // frame * frame
            p = pred[:T].reshape(-1, frame)
            r = ref[:T].reshape(-1, frame)
            return (10 * torch.log10(
                r.pow(2).mean(-1) / ((p - r).pow(2).mean(-1) + eps) + eps
            )).clamp(-10, 35).mean().item()

        def lsd_single(pred, ref):
            window = torch.hann_window(512, device='cpu')
            def pspec(x):
                return torch.stft(x, n_fft=512, hop_length=128,
                                  window=window, return_complex=True,
                                  center=False, onesided=True
                                  ).abs().pow(2).clamp(min=1e-8)
            pp, pc = pspec(pred), pspec(ref)
            return torch.sqrt(
                (10 * (torch.log10(pc) - torch.log10(pp))).pow(2).mean(0)
            ).mean().item()

        sisnr_vals = []; sdr_vals = []; seg_vals = []; lsd_vals = []
        for i in range(n):
            p = pred_b[i].squeeze().float().cpu()
            c = clean_b[i].squeeze().float().cpu()
            noisy = noisy_b[i].squeeze().float().cpu()
            sisnr_vals.append(si_snr_single(p, c))
            sdr_vals.append(sdr_single(p, c) - sdr_single(noisy, c))
            seg_vals.append(seg_snr_single(p, c) - seg_snr_single(noisy, c))
            lsd_vals.append(lsd_single(p, c))

        def _pad(lst, total=4):
            return lst + [None] * (total - len(lst))

        sisnr_p = _pad(sisnr_vals)
        sdr_p   = _pad(sdr_vals[:2])   # only log first 2 for brevity
        seg_p   = _pad(seg_vals[:2])
        lsd_p   = _pad(lsd_vals[:2])

        pred_all = pred_b[:n].float().cpu()
        targ_all = clean_b[:n].float().cpu()

        epochs_done = _read_epochs_done()

        row = {
            'epoch':           epochs_done + 1,
            'sisnr_sample_0':  f"{sisnr_p[0]:.4f}" if sisnr_p[0] is not None else '',
            'sisnr_sample_1':  f"{sisnr_p[1]:.4f}" if sisnr_p[1] is not None else '',
            'sisnr_sample_2':  f"{sisnr_p[2]:.4f}" if sisnr_p[2] is not None else '',
            'sisnr_sample_3':  f"{sisnr_p[3]:.4f}" if sisnr_p[3] is not None else '',
            'sisnr_mean':      f"{sum(sisnr_vals)/len(sisnr_vals):.4f}",
            'sdr_sample_0':    f"{sdr_p[0]:.4f}"   if sdr_p[0] is not None else '',
            'sdr_sample_1':    f"{sdr_p[1]:.4f}"   if sdr_p[1] is not None else '',
            'sdr_mean':        f"{sum(sdr_vals)/len(sdr_vals):.4f}",
            'seg_snr_sample_0':f"{seg_p[0]:.4f}"   if seg_p[0] is not None else '',
            'seg_snr_sample_1':f"{seg_p[1]:.4f}"   if seg_p[1] is not None else '',
            'seg_snr_mean':    f"{sum(seg_vals)/len(seg_vals):.4f}",
            'lsd_sample_0':    f"{lsd_p[0]:.4f}"   if lsd_p[0] is not None else '',
            'lsd_sample_1':    f"{lsd_p[1]:.4f}"   if lsd_p[1] is not None else '',
            'lsd_mean':        f"{sum(lsd_vals)/len(lsd_vals):.4f}",
            'pred_min':        f"{pred_all.min().item():.4f}",
            'pred_max':        f"{pred_all.max().item():.4f}",
            'pred_power_mean': f"{pred_all.pow(2).mean().item():.4f}",
            'targ_power_mean': f"{targ_all.pow(2).mean().item():.4f}",
            'power_ratio':     f"{(pred_all.pow(2).mean() / (targ_all.pow(2).mean() + 1e-8)).item():.4f}",
        }

        with open(self._log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self._log_fields).writerow(row)

    def after_batch(self):
        # capture first validation batch as fixed diagnostic samples
        if not self.training and self._fixed_batch is None:
            try:
                noisy = self.learn.xb[0].detach().cpu()
                clean = self.learn.yb[0].detach().cpu()
                pred  = self.learn.pred.detach().cpu()
                self._fixed_batch = (noisy, clean, pred)
            except Exception:
                pass

In [ ]:
class PeriodicPESQSTOI(Callback):
    """
    Computes PESQ and STOI on CPU every N epochs on a small validation subset.
    Never crashes training - all errors are caught and logged.
    
    PESQ target: > 2.5
    STOI target: > 0.88
    """
    # after EpochTracker(60) and Recorder(50)

    order = 70

    def __init__(self, fname='training_stats/pesq_stoi_log.csv',
                 every_n=10, n_batches=10):
        self.fname = fname
        self.every_n = every_n
        self.n_batches = n_batches
        self._init_file()

    def _init_file(self):
        os.makedirs(os.path.dirname(self.fname), exist_ok=True)
        if not Path(self.fname).exists():
            with open(self.fname, 'w', newline='') as f:
                csv.writer(f).writerow([
                    'epoch',
                    'pesq_mean', 'pesq_min', 'pesq_max',
                    'stoi_mean', 'stoi_min', 'stoi_max',
                    'n_samples', 'elapsed_seconds'
                ])

    def before_epoch(self):
        self._epoch_start = time.time()

    def after_epoch(self):
        if self._epoch_start is None:
            return
        if (time.time() - self._epoch_start) < 5.0:
            return

        epoch_num = _read_epochs_done()

        if epoch_num % self.every_n != 0:
            return

        t_start = time.time()

        try:
            from pesq import pesq as pesq_fn
            from pystoi import stoi as stoi_fn
        except ImportError as e:
            print(f"\n[PeriodicPESQSTOI] Import error: {e} - skipping")
            return

        try:
            self.learn.model.eval()
            pesq_scores = []
            stoi_scores = []

            for batch_idx, (xb, yb) in enumerate(self.learn.dls.valid):
                if batch_idx >= self.n_batches:
                    break

                with torch.no_grad():
                    pred = self.learn.model(xb)

                pred_np = pred.squeeze(1).detach().float().cpu().numpy()
                targ_np = yb.squeeze(1).detach().float().cpu().numpy()

                del pred

                for i in range(len(pred_np)):
                    try:
                        pesq_scores.append(
                            pesq_fn(16000, targ_np[i], pred_np[i], 'wb')
                        )
                    except Exception:
                        pass
                    try:
                        stoi_scores.append(
                            stoi_fn(targ_np[i], pred_np[i],
                                    16000, extended=False)
                        )
                    except Exception:
                        pass

                del pred_np, targ_np

            elapsed   = time.time() - t_start
            pesq_mean = float(np.mean(pesq_scores)) if pesq_scores else 0.0
            pesq_min  = float(np.min(pesq_scores))  if pesq_scores else 0.0
            pesq_max  = float(np.max(pesq_scores))  if pesq_scores else 0.0
            stoi_mean = float(np.mean(stoi_scores)) if stoi_scores else 0.0
            stoi_min  = float(np.min(stoi_scores))  if stoi_scores else 0.0
            stoi_max  = float(np.max(stoi_scores))  if stoi_scores else 0.0
            n_samples = len(pesq_scores)

            print(f"\n[Epoch {epoch_num}] PESQ={pesq_mean:.4f} "
                  f"(min={pesq_min:.4f} max={pesq_max:.4f}) | "
                  f"STOI={stoi_mean:.4f} "
                  f"(min={stoi_min:.4f} max={stoi_max:.4f}) | "
                  f"n={n_samples} | {elapsed:.1f}s")

            pesq_target = "✓" if pesq_mean > 2.5  else "✗"
            stoi_target = "✓" if stoi_mean > 0.88 else "✗"
            print(f"           PESQ target >2.5:  {pesq_target} | "
                  f"STOI target >0.88: {stoi_target}")

            with open(self.fname, 'a', newline='') as f:
                csv.writer(f).writerow([
                    epoch_num,
                    f"{pesq_mean:.4f}", f"{pesq_min:.4f}", f"{pesq_max:.4f}",
                    f"{stoi_mean:.4f}", f"{stoi_min:.4f}", f"{stoi_max:.4f}",
                    n_samples, f"{elapsed:.1f}"
                ])

        except Exception as e:
            elapsed = time.time() - t_start
            print(f"\n[PeriodicPESQSTOI] Error at epoch {epoch_num}: {e}")
            with open(self.fname, 'a', newline='') as f:
                csv.writer(f).writerow([
                    epoch_num, f"ERROR: {e}",
                    '', '', '', '', '', '', f"{elapsed:.1f}"
                ])

        finally:
            pesq_scores = []
            stoi_scores = []
            import gc
            gc.collect()
            self.learn.model.train()

In [ ]:
class TrainingHealthCallback(Callback):
    """
    Monitors training health every N epochs.
    Catches: frozen weights, gradient issues, optimizer staleness,
    EPOCH_SEED not advancing, model output collapse.
    """
    order = 82   # after metrics, before RecorderCleaner

    def __init__(self, every_n=5, log_path='training_stats/health_log.csv'):
        self.every_n  = every_n
        self.log_path = Path(log_path)
        self._prev_param_hash = None
        self._prev_epoch_seed = None

    def before_fit(self):
        self.log_path.parent.mkdir(exist_ok=True)
        if not self.log_path.exists():
            with open(self.log_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=self._fields())
                writer.writeheader()

    def _fields(self):
        return [
            'epoch',
            'epoch_seed',
            'epoch_seed_advanced',
            'weights_changed',
            'max_weight_delta',
            'grad_norm_mean',
            'grad_norm_max',
            'nan_grads',
            'zero_grads',
            'output_rms',
            'target_rms',
            'power_ratio_db',
            'loss_value',
            'current_lr',
        ]

    # def after_batch(self):
    #     if self.training:
    #         self._batches_this_epoch = getattr(self, '_batches_this_epoch', 0) + 1

    def before_epoch(self):
        self._start_time = time.time()

    def after_epoch(self):

        elapsed = time.time() - self._start_time

        if elapsed < 5.0:
            return

        epochs_done = _read_epochs_done()
        if epochs_done % self.every_n != 0:
            return

        row = {'epoch': epochs_done}

        # EPOCH_SEED check
        current_seed = EPOCH_SEED.value
        row['epoch_seed'] = current_seed
        if self._prev_epoch_seed is not None:
            row['epoch_seed_advanced'] = (current_seed != self._prev_epoch_seed)
            if not row['epoch_seed_advanced']:
                print(f"[HealthCheck ep{self.learn.epoch}] "
                      f"⚠ EPOCH_SEED did not advance "
                      f"(stuck at {current_seed})")
        else:
            row['epoch_seed_advanced'] = True
        self._prev_epoch_seed = current_seed

        # weight change check - hash a subset of params for speed
        try:
            param_vals = []
            for name, p in self.learn.model.named_parameters():
                if p.requires_grad:
                    param_vals.append(p.data.mean().item())
                    if len(param_vals) >= 20:
                        break
            current_hash = hash(tuple(round(v, 8) for v in param_vals))

            if self._prev_param_hash is not None:
                weights_changed = (current_hash != self._prev_param_hash)
                row['weights_changed'] = weights_changed
                if not weights_changed:
                    print(f"[HealthCheck ep{self.learn.epoch}] "
                          f"✗ WEIGHTS DID NOT CHANGE since last check")
            else:
                row['weights_changed'] = True

            self._prev_param_hash = current_hash

            # max delta across a few key params
            deltas = [abs(v) for v in param_vals]
            row['max_weight_delta'] = round(max(deltas), 8)

        except Exception as e:
            row['weights_changed']   = 'error'
            row['max_weight_delta']  = 'error'

        # gradient stats from last backward
        try:
            grad_norms = []
            nan_grads  = 0
            zero_grads = 0
            for p in self.learn.model.parameters():
                if p.grad is None:
                    continue
                if not torch.isfinite(p.grad).all():
                    nan_grads += 1
                    continue
                gnorm = p.grad.norm().item()
                if gnorm < 1e-10:
                    zero_grads += 1
                else:
                    grad_norms.append(gnorm)

            row['grad_norm_mean'] = round(
                sum(grad_norms)/len(grad_norms), 6) if grad_norms else 0.0
            row['grad_norm_max']  = round(max(grad_norms), 6) if grad_norms else 0.0
            row['nan_grads']      = nan_grads
            row['zero_grads']     = zero_grads

            if nan_grads > 0:
                print(f"[HealthCheck ep{self.learn.epoch}] "
                      f"✗ {nan_grads} NaN gradients")

        except Exception as e:
            row['grad_norm_mean'] = 'error'
            row['grad_norm_max']  = 'error'
            row['nan_grads']      = 'error'
            row['zero_grads']     = 'error'

        # output vs target level
        try:
            xb, yb = self.learn.dls.valid.one_batch()
            with torch.no_grad():
                pred = self.learn.model(xb)
            pred_rms = pred.float().pow(2).mean().sqrt().item()
            targ_rms = yb.float().pow(2).mean().sqrt().item()
            ratio_db = 20 * np.log10(pred_rms / (targ_rms + 1e-8))

            row['output_rms']      = round(pred_rms, 6)
            row['target_rms']      = round(targ_rms, 6)
            row['power_ratio_db']  = round(ratio_db, 4)

            if abs(ratio_db) > 3.0:
                print(f"[HealthCheck ep{self.learn.epoch}] "
                      f"⚠ Level mismatch: {ratio_db:+.1f} dB "
                      f"(pred_rms={pred_rms:.4f}, targ_rms={targ_rms:.4f})")

        except Exception as e:
            row['output_rms']     = 'error'
            row['target_rms']     = 'error'
            row['power_ratio_db'] = 'error'

        # current loss and lr
        try:
            if self.learn.recorder.values:
                row['loss_value'] = round(
                    self.learn.recorder.values[-1][1], 6)
            else:
                row['loss_value'] = ''
            row['current_lr'] = self.learn.opt.hypers[-1].get('lr', '')
        except Exception:
            row['loss_value'] = ''
            row['current_lr'] = ''

        # write row
        with open(self.log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self._fields()).writerow(row)

        # concise console summary
        seed_ok    = '✓' if row.get('epoch_seed_advanced', True) else '✗'
        weights_ok = '✓' if row.get('weights_changed', True) else '✗'
        grads_ok   = '✓' if row.get('nan_grads', 0) == 0 else '✗'
        level_db   = row.get('power_ratio_db', 0)
        level_ok   = '✓' if isinstance(level_db, float) and abs(level_db) < 3 else '⚠'

        print(
            f"[HealthCheck ep{self.learn.epoch}] "
            f"seed={seed_ok} weights={weights_ok} grads={grads_ok} "
            f"level={level_ok}({level_db:+.1f}dB) "
            f"grad_norm={row.get('grad_norm_mean', 'N/A'):.4f}"
        )

In [ ]:
class BatchTimingCallback(Callback):
    order = 0

    def __init__(self, n_batches=100, report_every=25, 
                 out_path='training_stats/batch_timing.csv'):
        self.n_batches    = n_batches
        self.report_every = report_every
        self.out_path     = Path(out_path)
        self._rows        = []   # accumulated in memory
        self._batch_count = 0
        self._times       = defaultdict(list)

    def before_fit(self):
        self.out_path.parent.mkdir(parents=True, exist_ok=True)

    def before_batch(self):
        self._t_batch = time.perf_counter()
        self._t_phase = time.perf_counter()

    def after_pred(self):
        self._times['1_forward'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_loss(self):
        self._times['2_loss_compute'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_backward(self):
        self._times['3_backward'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_step(self):
        self._times['4_optimizer_step'].append(time.perf_counter() - self._t_phase)
        self._t_phase = time.perf_counter()

    def after_batch(self):
        full = time.perf_counter() - self._t_batch
        self._times['0_full_batch'].append(full)
        self._batch_count += 1

        self._rows.append({
            'batch':        self._batch_count,
            'full_ms':      full * 1000,
            'forward_ms':   self._times['1_forward'][-1]     * 1000 if self._times['1_forward']     else 0,
            'loss_ms':      self._times['2_loss_compute'][-1] * 1000 if self._times['2_loss_compute'] else 0,
            'backward_ms':  self._times['3_backward'][-1]    * 1000 if self._times['3_backward']    else 0,
            'optimizer_ms': self._times['4_optimizer_step'][-1] * 1000 if self._times['4_optimizer_step'] else -1,
            # -1 means step was skipped this batch (GradientAccumulation)
        })

        if self._batch_count % self.report_every == 0:
            self._print_summary()

        if self._batch_count >= self.n_batches:
            self._print_summary()
            self._flush_csv()
            self.learn.remove_cb(self)

    def after_fit(self):
        # safety flush if training ends before n_batches reached
        if self._rows:
            self._flush_csv()

    def _flush_csv(self):
        import csv
        write_header = not self.out_path.exists()
        with open(self.out_path, 'a', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=self._rows[0].keys())
            if write_header:
                writer.writeheader()
            writer.writerows(self._rows)
        self._rows = []  # free memory after flush
        print(f"[BatchTimingCallback] flushed to {self.out_path}")

    def _print_summary(self):
        print(f"\n[BatchTimingCallback] after {self._batch_count} batches:")
        print(f"  {'Phase':<25} {'avg(ms)':>10} {'max(ms)':>10} {'total(s)':>10}")
        print(f"  {'-'*50}")
        for phase, times in sorted(self._times.items()):
            print(f"  {phase:<25} "
                  f"{np.mean(times)*1000:>10.1f} "
                  f"{np.max(times)*1000:>10.1f} "
                  f"{np.sum(times):>10.2f}")

In [ ]:
class CProfileCallback(Callback):
    """
    Runs cProfile over n_batches of training.
    
    Reports:
    1. Top functions by cumulative time
    2. Top functions by total self time  
    3. Full call chain for the single longest-running function
    4. Saves raw stats to disk for external analysis
    
    Usage:
        learn.add_cb(CProfileCallback(n_batches=50, out_dir='training_stats/cprofile'))
    """
    order = 0

    def __init__(self, n_batches=50, top_n=30, 
                 out_dir='training_stats/cprofile'):
        self.n_batches  = n_batches
        self.top_n      = top_n
        self.out_dir    = Path(out_dir)
        self._batch     = 0
        self._profiler  = None
        self._done      = False

    def before_fit(self):
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self._profiler = cProfile.Profile()
        self._profiler.enable()
        print(f"[CProfileCallback] profiling started - "
              f"will capture {self.n_batches} batches")

    def after_batch(self):
        if self._done:
            return
        self._batch += 1
        if self._batch >= self.n_batches:
            self._profiler.disable()
            self._done = True
            print(f"[CProfileCallback] {self.n_batches} batches captured - "
                  f"generating reports...")
            self._report()
            self.learn.remove_cb(self)

    def after_fit(self):
        # safety - disable if training ends before n_batches
        if not self._done and self._profiler is not None:
            self._profiler.disable()
            self._report()

    def _report(self):
        stats = pstats.Stats(self._profiler)

        # -- Save raw stats to disk for snakeviz or other tools ----------
        raw_path = self.out_dir / 'profile.stats'
        stats.dump_stats(str(raw_path))
        print(f"[CProfileCallback] raw stats saved to {raw_path}")
        print(f"  (view with: snakeviz {raw_path}  or  "
              f"python -m pstats {raw_path})\n")

        # -- Report 1: top by cumulative time ----------------------------
        self._print_section(
            "TOP FUNCTIONS BY CUMULATIVE TIME "
            "(includes time in callees - best for finding slow call chains)",
            stats, sort='cumulative'
        )

        # -- Report 2: top by self time -----------------------------------
        self._print_section(
            "TOP FUNCTIONS BY SELF TIME "
            "(excludes callees - best for finding actual bottleneck)",
            stats, sort='tottime'
        )

        # -- Report 3: full call chain for the top cumulative entry -------
        self._print_call_chain(stats)

        # -- Report 4: per-phase breakdown --------------------------------
        self._print_phase_breakdown(stats)

    def _print_section(self, title, stats, sort):
        s   = io.StringIO()
        ps  = pstats.Stats(self._profiler, stream=s)
        ps.sort_stats(sort)
        ps.print_stats(self.top_n)
        
        out_path = self.out_dir / f'report_{sort}.txt'
        report   = s.getvalue()
        
        with open(out_path, 'w') as f:
            f.write(f"{'='*70}\n{title}\n{'='*70}\n")
            f.write(report)
        
        print(f"\n{'='*70}")
        print(title)
        print('='*70)
        # print first 60 lines to notebook
        lines = report.split('\n')
        print('\n'.join(lines[:60]))
        if len(lines) > 60:
            print(f"  ... ({len(lines)-60} more lines in {out_path})")

    def _print_call_chain(self, stats):
        """
        Find the single function with highest cumulative time,
        then print its full caller and callee chain.
        """
        s  = io.StringIO()
        ps = pstats.Stats(self._profiler, stream=s)
        ps.sort_stats('cumulative')
        
        # get top entry from internal stats dict
        # stats.stats format: {(file,line,func): (cc, nc, tt, ct, callers)}
        #   cc=primitive calls, nc=total calls, tt=self time, ct=cumul time
        raw   = ps.stats
        top_entry = max(raw.items(), key=lambda x: x[1][3])  # sort by ct
        top_key   = top_entry[0]   # (file, line, func)
        top_data  = top_entry[1]   # (cc, nc, tt, ct, callers)

        file_, line, func = top_key
        cc, nc, tt, ct, callers = top_data

        report_lines = []
        report_lines.append(f"\n{'='*70}")
        report_lines.append("FULL CALL CHAIN FOR SLOWEST FUNCTION")
        report_lines.append('='*70)
        report_lines.append(
            f"Function : {func}\n"
            f"Location : {file_}:{line}\n"
            f"Calls    : {nc} ({cc} primitive)\n"
            f"Self time: {tt*1000/max(nc,1):.2f} ms/call  "
            f"({tt:.3f}s total)\n"
            f"Cum time : {ct*1000/max(nc,1):.2f} ms/call  "
            f"({ct:.3f}s total)"
        )

        # -- callers of top function --------------------------------------
        report_lines.append(f"\n-- CALLED BY ({len(callers)} callers) --")
        sorted_callers = sorted(
            callers.items(),
            key=lambda x: x[1][3],  # sort by cumulative time
            reverse=True
        )
        for (c_file, c_line, c_func), (c_cc, c_nc, c_tt, c_ct) in sorted_callers[:10]:
            report_lines.append(
                f"  {c_func:<45} "
                f"calls={c_nc:>6}  "
                f"cum={c_ct*1000/max(c_nc,1):>8.2f}ms/call  "
                f"@ {c_file}:{c_line}"
            )

        # -- what top function calls --------------------------------------
        report_lines.append(f"\n-- CALLS INTO --")
        callees = {
            key: data for key, data in raw.items()
            if top_key in data[4]   # data[4] is callers dict
        }
        sorted_callees = sorted(
            callees.items(),
            key=lambda x: x[1][3],
            reverse=True
        )
        for (e_file, e_line, e_func), (e_cc, e_nc, e_tt, e_ct, _) in sorted_callees[:15]:
            report_lines.append(
                f"  {e_func:<45} "
                f"calls={e_nc:>6}  "
                f"self={e_tt*1000/max(e_nc,1):>8.2f}ms/call  "
                f"cum={e_ct*1000/max(e_nc,1):>8.2f}ms/call  "
                f"@ {e_file}:{e_line}"
            )

        # -- grandcallees (one level deeper) -----------------------------
        report_lines.append(f"\n-- GRANDCALLEES (next level) --")
        grandcallees = {}
        for callee_key in list(callees.keys())[:5]:  # top 5 callees only
            for key, data in raw.items():
                if callee_key in data[4]:
                    grandcallees[key] = data

        sorted_grand = sorted(
            grandcallees.items(),
            key=lambda x: x[1][3],
            reverse=True
        )
        for (g_file, g_line, g_func), (g_cc, g_nc, g_tt, g_ct, _) in sorted_grand[:15]:
            report_lines.append(
                f"  {g_func:<45} "
                f"calls={g_nc:>6}  "
                f"self={g_tt*1000/max(g_nc,1):>8.2f}ms/call  "
                f"cum={g_ct*1000/max(g_nc,1):>8.2f}ms/call  "
                f"@ {g_file}:{g_line}"
            )

        report = '\n'.join(report_lines)
        out_path = self.out_dir / 'call_chain.txt'
        with open(out_path, 'w') as f:
            f.write(report)
        print(report)
        print(f"\n  (saved to {out_path})")

    def _print_phase_breakdown(self, stats):
        """
        Aggregate time by module/phase - fastai, torch, model, loss, etc.
        Groups entries by filename prefix for a high-level view.
        """
        raw = stats.stats
        
        phase_totals = {}
        for (file_, line, func), (cc, nc, tt, ct, callers) in raw.items():
            # categorize by file path
            if 'fastai'        in file_: phase = 'fastai'
            elif 'torch'       in file_: phase = 'torch'
            elif 'directml'    in file_: phase = 'directml'
            elif 'ipykernel'   in file_: phase = 'your_code (ipykernel)'
            elif 'pyroomacous' in file_: phase = 'pyroomacoustics'
            elif 'numpy'       in file_: phase = 'numpy'
            elif 'scipy'       in file_: phase = 'scipy'
            elif '<'           in file_: phase = 'builtins/C_extensions'
            else:                        phase = f'other: {Path(file_).name}'

            if phase not in phase_totals:
                phase_totals[phase] = {'tt': 0, 'ct': 0, 'calls': 0}
            phase_totals[phase]['tt']    += tt
            phase_totals[phase]['ct']    += ct
            phase_totals[phase]['calls'] += nc

        report_lines = [
            f"\n{'='*70}",
            "TIME BY MODULE/PHASE",
            '='*70,
            f"{'Module':<35} {'self(s)':>10} {'cum(s)':>10} {'calls':>10}",
            '-'*65
        ]

        for phase, data in sorted(
            phase_totals.items(), 
            key=lambda x: -x[1]['tt']
        ):
            report_lines.append(
                f"{phase:<35} "
                f"{data['tt']:>10.3f} "
                f"{data['ct']:>10.3f} "
                f"{data['calls']:>10,}"
            )

        report = '\n'.join(report_lines)
        out_path = self.out_dir / 'phase_breakdown.txt'
        with open(out_path, 'w') as f:
            f.write(report)
        print(report)
        print(f"\n  (saved to {out_path})")

In [ ]:
class RecorderCleaner(Callback):
    order = 80
    def after_epoch(self):
        if len(self.learn.recorder.log) < 3:
            return
        r = self.learn.recorder
        n_losses = len(r.losses)
        n_lrs    = len(r.lrs)
        n_iters  = len(r.iters)
        r.losses.clear()
        r.lrs.clear()
        # Keep iters in sync with losses/lrs - clear all but append
        # a sentinel 0 so smooth_loss.count is still readable if needed
        r.iters.clear()
        print(f"[RecorderCleaner] epoch {self.learn.epoch} - "
              f"cleared losses={n_losses}  lrs={n_lrs}  iters={n_iters}")

    def after_fit(self):
        r = self.learn.recorder
        r.losses.clear()
        r.lrs.clear()
        r.iters.clear()

In [ ]:
class DMLQueueFlush(Callback):
    order = 99  # after everything else
    def after_epoch(self):
        # Force DML to flush its command queue - resets queue depth
        # so next epoch starts clean rather than inheriting accumulated depth
        try:
            _ = torch.tensor(1.0, device=device).item()  # force sync
        except Exception:
            pass

In [ ]:
def _read_epochs_done():
    """Read current epochs_done from tracker file"""
    try:
        with open('training_stats/epoch_tracker.json', 'r') as f:
            return int(json.loads(f.read().strip()).get('epochs_done'))
    except:
        return 0

In [ ]:
def set_epoch_seed(epoch):
    """Call before each epoch from main process"""
    EPOCH_SEED.value = epoch

    try:
        torch_directml.PrivateUse1Module.manual_seed_all(epoch)
    except Exception:
        pass